In [ ]:
import gc
import sys
import numpy as np
import pandas as pd
import faiss
from pathlib import Path
from sentence_transformers import SentenceTransformer

# Paths
SPLITS_TRAIN   = Path("splits/config_id=baseline/train/")
OUTPUT_DIR     = Path("semantic/")
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints/"

# Hyperparams
MODEL_NAME    = "sentence-transformers/all-MiniLM-L6-v2"
ENCODE_BATCH  = 32

# item_semantic_profile
W_ITEM_TEXT   = 0.6
W_ITEM_REVIEW = 0.4

# item_review_profile filter
MIN_RELIABILITY = 0.3
MIN_TEXT_LEN    = 50

# user_semantic_profile
BETA      = 0.6    # weight cho recent_profile
LAMBDA    = 0.01   # recency decay: exp(-lambda * days_ago)
N_RECENT  = 20     # top-N recent items cho recent_profile


# Helpers
def l2_normalize(v: np.ndarray) -> np.ndarray:
    norm = np.linalg.norm(v)
    return v / (norm + 1e-9)

def get_parquet_files(folder: Path) -> list:
    files = sorted(folder.glob("**/*.parquet"))
    if not files:
        raise FileNotFoundError(f"Không tìm thấy .parquet trong {folder}")
    return files

In [ ]:
print("BƯỚC 7.1 — Item semantic profiles + FAISS")

OUTPUT_DIR.mkdir(exist_ok=True)
CHECKPOINT_DIR.mkdir(exist_ok=True)
model    = SentenceTransformer(MODEL_NAME, device="cuda")
dim      = model.get_sentence_embedding_dimension()
index    = faiss.IndexFlatIP(dim)
item_ids = []
seen     = set()

pq_files = get_parquet_files(SPLITS_TRAIN)
print(f"  Tổng file parquet: {len(pq_files)}")

for file_idx, pq_file in enumerate(pq_files):
    ckpt_emb = CHECKPOINT_DIR / f"item_profiles_{file_idx:04d}.npy"
    ckpt_ids = CHECKPOINT_DIR / f"item_ids_{file_idx:04d}.parquet"

    if ckpt_emb.exists() and ckpt_ids.exists():
        print(f"  [{file_idx:04d}] checkpoint có → load và add vào FAISS")
        embs = np.load(ckpt_emb)
        ids  = pd.read_parquet(ckpt_ids)["parent_asin"].tolist()
        if len(embs) > 0:
            index.add(embs.astype("float32"))
        item_ids.extend(ids)
        seen.update(ids)
        del embs, ids; gc.collect()
        continue

    print(f"\n  [{file_idx:04d}] {pq_file.name}")

    # Load text fields
    df_text = pd.read_parquet(
        pq_file,
        columns=["parent_asin", "item_title", "brand", "main_category",  "description", "features"]
    )
    df_text = df_text.drop_duplicates("parent_asin")
    df_text = df_text[~df_text["parent_asin"].isin(seen)].reset_index(drop=True)

    if len(df_text) == 0:
        print(f"    Không có item mới.")
        np.save(ckpt_emb, np.zeros((0, dim), dtype="float32"))
        pd.DataFrame({"parent_asin": []}).to_parquet(ckpt_ids, index=False)
        continue

    df_text["item_text"] = (
        df_text["item_title"].fillna("") + " " +
        df_text["brand"].fillna("") + " " +
        df_text["main_category"].fillna("") + " " +
        df_text["description"].fillna("") + " " +
        df_text["features"].fillna("")
        ).str.strip()
    df_text = df_text[df_text["item_text"].str.len() > 0].reset_index(drop=True)
    print(f"    Item mới: {len(df_text):,}")

    # Encode item_text_emb
    text_embs = model.encode(
        df_text["item_text"].tolist(),
        batch_size=ENCODE_BATCH,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32")

    # Load review_text của file này, filter noise
    df_rev = pd.read_parquet(
        pq_file,
        columns=["parent_asin", "review_text", "text_len",
                 "reliability_score", "strength_score"]
    )
    df_rev = df_rev[
        df_rev["parent_asin"].isin(set(df_text["parent_asin"])) &
        (df_rev["reliability_score"] > MIN_RELIABILITY) &
        (df_rev["text_len"] > MIN_TEXT_LEN)
    ].reset_index(drop=True)
    print(f"    Review sau filter: {len(df_rev):,}")

    # Encode review_text + pool theo item
    item_review_map = {}

    if len(df_rev) > 0:
        rev_embs = model.encode(
            df_rev["review_text"].fillna("").tolist(),
            batch_size=ENCODE_BATCH,
            show_progress_bar=True,
            normalize_embeddings=True,
            convert_to_numpy=True,
        ).astype("float32")

        df_rev = df_rev.reset_index(drop=True)
        df_rev["_eidx"] = range(len(df_rev))

        for asin, grp in df_rev.groupby("parent_asin"):
            idxs   = grp["_eidx"].values
            w      = grp["reliability_score"].values * grp["strength_score"].values
            w      = w / (w.sum() + 1e-9)
            pooled = (rev_embs[idxs] * w[:, None]).sum(axis=0)
            item_review_map[asin] = l2_normalize(pooled)

        del rev_embs; gc.collect()

    del df_rev; gc.collect()

    # Blend: item_semantic_profile
    asin_list     = df_text["parent_asin"].tolist()
    item_profiles = np.zeros((len(asin_list), dim), dtype="float32")

    for j, asin in enumerate(asin_list):
        text_vec   = text_embs[j]
        review_vec = item_review_map.get(asin)

        if review_vec is not None:
            blended = W_ITEM_TEXT * text_vec + W_ITEM_REVIEW * review_vec
        else:
            # Không có review đủ chất lượng → fallback chỉ dùng text
            blended = text_vec

        item_profiles[j] = l2_normalize(blended)

    # Add vào FAISS + lưu checkpoint
    index.add(item_profiles)
    item_ids.extend(asin_list)
    seen.update(asin_list)

    np.save(ckpt_emb, item_profiles)
    pd.DataFrame({"parent_asin": asin_list}).to_parquet(ckpt_ids, index=False)
    print(f"    Saved checkpoint item_profiles_{file_idx:04d}")

    del df_text, text_embs, item_review_map, item_profiles; gc.collect()

# Lưu FAISS + item_id_map
print(f"\n  Tổng item trong FAISS: {index.ntotal:,}")
faiss.write_index(index, str(OUTPUT_DIR / "faiss_index_v1.index"))
pd.DataFrame({
    "parent_asin": item_ids,
    "faiss_pos":   range(len(item_ids))
}).to_parquet(OUTPUT_DIR / "item_id_map_v1.parquet", index=False)

del index; gc.collect()
print("Bước 7.1 hoàn thành!")


In [ ]:
print("BƯỚC 7.2 — Encode review_text cho user profiles")

OUTPUT_DIR.mkdir(exist_ok=True)
CHECKPOINT_DIR.mkdir(exist_ok=True)
model = SentenceTransformer(MODEL_NAME, device="cuda")
dim   = model.get_sentence_embedding_dimension()

COLS = ["reviewer_id", "parent_asin", "review_text", "text_len",
        "reliability_score", "strength_score", "review_time",
        "bpr_role", "interaction_type"]

pq_files   = get_parquet_files(SPLITS_TRAIN)
meta_parts = []
global_idx = 0

for file_idx, pq_file in enumerate(pq_files):
    ckpt_emb  = CHECKPOINT_DIR / f"user_rev_{file_idx:04d}_emb.npy"
    ckpt_meta = CHECKPOINT_DIR / f"user_rev_{file_idx:04d}_meta.parquet"

    if ckpt_emb.exists() and ckpt_meta.exists():
        print(f"  [{file_idx:04d}] checkpoint có → bỏ qua")
        saved = pd.read_parquet(ckpt_meta)
        meta_parts.append(saved)
        global_idx += len(saved)
        del saved; gc.collect()
        continue

    print(f"\n  [{file_idx:04d}] {pq_file.name}")
    df = pd.read_parquet(pq_file, columns=COLS)

    df_pos = df[
        (df["bpr_role"] == "positive") &
        (df["reliability_score"] > MIN_RELIABILITY) &
        (df["text_len"] > MIN_TEXT_LEN) &
        (df["interaction_type"].isin(["strong_positive", "weak_positive"]))
    ].copy().reset_index(drop=True)
    del df; gc.collect()

    if len(df_pos) == 0:
        print(f"    Không có row hợp lệ.")
        np.save(ckpt_emb, np.zeros((0, dim), dtype="float32"))
        pd.DataFrame(columns=["reviewer_id", "parent_asin", "emb_idx",
                               "reliability_score", "strength_score",
                               "review_time"]).to_parquet(ckpt_meta, index=False)
        continue

    print(f"    Rows sau filter: {len(df_pos):,}")

    embs = model.encode(
        df_pos["review_text"].fillna("").tolist(),
        batch_size=ENCODE_BATCH,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32")

    df_pos["emb_idx"] = range(global_idx, global_idx + len(df_pos))
    global_idx += len(df_pos)

    meta = df_pos[["reviewer_id", "parent_asin", "emb_idx",
                    "reliability_score", "strength_score", "review_time"]].copy()
    np.save(ckpt_emb, embs)
    meta.to_parquet(ckpt_meta, index=False)
    meta_parts.append(meta)

    del df_pos, embs, meta; gc.collect()
    print(f"    Saved user_rev_{file_idx:04d}_*")

print("\n  Gộp review_meta...")
pd.concat(meta_parts, ignore_index=True) \
  .to_parquet(OUTPUT_DIR / "review_meta.parquet", index=False)

#done_flag.touch()
print("Bước 7.2 hoàn thành!")


In [ ]:
print("BƯỚC 7.3 — Build user semantic profiles (checkpoint + batch)")
print("="*60)
USER_BATCH_SIZE = 50_000   # số user xử lý mỗi lần, giảm nếu vẫn tràn RAM

USER_CKPT_DIR = OUTPUT_DIR / "user_ckpts/"
USER_CKPT_DIR.mkdir(exist_ok=True)
# [1/4] Build item lookup: asin → numpy row index
# Gộp tất cả item_ids_XXXX.parquet → biết asin nằm ở chunk nào, row nào
print("\n[1/4] Build item lookup từ checkpoint 7.1...")
ckpt_id_files  = sorted(CHECKPOINT_DIR.glob("item_ids_*.parquet"))
ckpt_emb_files = sorted(CHECKPOINT_DIR.glob("item_profiles_*.npy"))
assert len(ckpt_id_files) == len(ckpt_emb_files), "Số checkpoint ids và emb không khớp!"

# Xây dựng: asin → (chunk_idx, row_idx_within_chunk)
asin_to_loc = {}   # {asin: (chunk_idx, local_row)}
for chunk_idx, id_file in enumerate(ckpt_id_files):
    ids = pd.read_parquet(id_file)["parent_asin"].tolist()
    for local_row, asin in enumerate(ids):
        asin_to_loc[asin] = (chunk_idx, local_row)

print(f"  Item lookup: {len(asin_to_loc):,} items từ {len(ckpt_id_files)} chunks")

# mmap array cho từng chunk — OS tự quản lý, không load vào RAM
mmap_chunks = [
    np.load(str(f), mmap_mode="r") for f in ckpt_emb_files
]
dim = mmap_chunks[0].shape[1]
print(f"  Embedding dim: {dim}")

def get_item_vec(asin: str):
    loc = asin_to_loc.get(asin)
    if loc is None:
        return None
    chunk_idx, local_row = loc
    # Slice mmap → chỉ đọc đúng 1 row từ disk
    return np.array(mmap_chunks[chunk_idx][local_row], dtype="float32")

# [2/4] Load review_meta + tính recency
print("\n[2/4] Load review_meta + tính recency_decay...")
df_meta = pd.read_parquet(OUTPUT_DIR / "review_meta.parquet")
print(f"  review_meta: {len(df_meta):,} rows")

sample = df_meta["review_time"].iloc[0]
if isinstance(sample, (int, float, np.integer, np.floating)):
    df_meta["review_time"] = pd.to_datetime(df_meta["review_time"], unit="s")
else:
    df_meta["review_time"] = pd.to_datetime(df_meta["review_time"])

max_date = df_meta["review_time"].max()
print(f"  Mốc cuối train: {max_date.date()}")

df_meta["days_ago"]      = (max_date - df_meta["review_time"]).dt.days.clip(lower=0)
df_meta["recency_decay"] = np.exp(-LAMBDA * df_meta["days_ago"])

# [3/4] Xử lý user theo batch, checkpoint sau mỗi batch
print(f"\n[3/4] Build profiles (BETA={BETA}, λ={LAMBDA}, N_RECENT={N_RECENT})...")
print(f"  Batch size: {USER_BATCH_SIZE:,} users/batch")

# Tìm batch đã xong để resume
done_batch_files = sorted(USER_CKPT_DIR.glob("batch_*.parquet"))
done_batches     = {int(f.stem.split("_")[1]) for f in done_batch_files}
if done_batches:
    print(f"  Resume: đã xong {len(done_batches)} batch, tiếp tục từ batch {max(done_batches)+1}")

all_user_ids = sorted(df_meta["reviewer_id"].unique())
total_users  = len(all_user_ids)
print(f"  Tổng users: {total_users:,}")

n_batches = (total_users + USER_BATCH_SIZE - 1) // USER_BATCH_SIZE

for batch_idx in range(n_batches):
    if batch_idx in done_batches:
        print(f"  Batch {batch_idx:04d}/{n_batches} → đã có checkpoint, bỏ qua")
        continue

    batch_start = batch_idx * USER_BATCH_SIZE
    batch_end   = min(batch_start + USER_BATCH_SIZE, total_users)
    batch_users = set(all_user_ids[batch_start:batch_end])

    print(f"\n  Batch {batch_idx:04d}/{n_batches} "
          f"(user {batch_start:,}–{batch_end:,})...")

    # Lọc df_meta chỉ lấy batch này — tránh groupby toàn bộ
    df_batch = df_meta[df_meta["reviewer_id"].isin(batch_users)].copy()

    batch_ids  = []
    batch_embs = []

    for user_id, grp in df_batch.groupby("reviewer_id"):
        # Lấy item vectors cho user này
        item_cache = {}
        for asin in grp["parent_asin"].unique():
            v = get_item_vec(asin)
            if v is not None:
                item_cache[asin] = v

        if not item_cache:
            continue

        valid = grp[grp["parent_asin"].isin(item_cache)]

        # Long-term: w = reliability * recency_decay
        lt_w  = valid["reliability_score"].values * valid["recency_decay"].values
        lt_w  = lt_w / (lt_w.sum() + 1e-9)
        lt_v  = np.stack([item_cache[a] for a in valid["parent_asin"]])
        long_term = (lt_v * lt_w[:, None]).sum(axis=0)

        # Recent: top-N gần nhất, w = strength_score
        recent_rows = valid.nlargest(N_RECENT, "review_time")
        rc_w  = recent_rows["strength_score"].values
        rc_w  = rc_w / (rc_w.sum() + 1e-9)
        rc_v  = np.stack([item_cache[a] for a in recent_rows["parent_asin"]])
        recent = (rc_v * rc_w[:, None]).sum(axis=0)

        # Blend + L2 normalize
        user_vec = BETA * recent + (1 - BETA) * long_term
        user_vec = l2_normalize(user_vec).astype("float32")

        batch_ids.append(user_id)
        batch_embs.append(user_vec)

    # Lưu checkpoint batch
    if batch_embs:
        batch_arr = np.stack(batch_embs)
        ckpt_path = USER_CKPT_DIR / f"batch_{batch_idx:04d}.parquet"
        # Lưu emb inline trong parquet dưới dạng list (nhỏ gọn, dễ gộp)
        pd.DataFrame({
            "reviewer_id": batch_ids,
            "embedding":   [v.tolist() for v in batch_arr]
        }).to_parquet(ckpt_path, index=False)
        print(f"    Saved {len(batch_ids):,} users → batch_{batch_idx:04d}.parquet")

    del df_batch, batch_ids, batch_embs; gc.collect()

# [4/4] Gộp tất cả batch checkpoint → file cuối
print("\n[4/4] Gộp tất cả batch checkpoints...")

batch_files = sorted(USER_CKPT_DIR.glob("batch_*.parquet"))

# Đếm tổng số user trước để pre-allocate array 1 lần
total = sum(len(pd.read_parquet(f, columns=["reviewer_id"])) for f in batch_files)
print(f"  Tổng users: {total:,}")

user_arr     = np.zeros((total, dim), dtype="float32")
reviewer_ids = []
cursor       = 0

for f in batch_files:
    df_part = pd.read_parquet(f)
    n = len(df_part)
    user_arr[cursor:cursor+n] = np.stack(
        df_part["embedding"].apply(np.array).values
    ).astype("float32")
    reviewer_ids.extend(df_part["reviewer_id"].tolist())
    cursor += n
    del df_part; gc.collect()

np.save(OUTPUT_DIR / "user_semantic_profiles_v1.npy", user_arr)
pd.DataFrame({
    "reviewer_id": reviewer_ids,
    "profile_pos": range(len(reviewer_ids))
}).to_parquet(OUTPUT_DIR / "user_id_map_v1.parquet", index=False)

print(f"  Saved: user_semantic_profiles_v1.npy  shape={user_arr.shape}")
print(f"  Saved: user_id_map_v1.parquet")
print("\nBước 7.3 hoàn thành!")
print("\nOutput trong semantic/:")

